In [ ]:
import wandb

api = wandb.Api()
# get all runs created today
runs = api.runs("learneddb/lcm", filters={"state": "finished"})

In [ ]:
from datetime import datetime

from tqdm import tqdm

runs_adjusted_ctr = 0
print(f"Found {len(runs)} runs")
for run in tqdm(runs):
    # get summary metrics
    summary = run.summary

    # retrieve and parse created_at
    created_at = run.created_at
    created_at = datetime.strptime(created_at, "%Y-%m-%dT%H:%M:%SZ")
    if created_at.date() < datetime.now().date():
        continue

    # compute new metrics
    selected_runtimes = []
    slowdowns = []
    try:
        for key in [
            "join_order_job_eval",
            "join_order_job_light_eval",
            "physop_eval",
            "physop_eval_with_idxs",
            "accesspath_eval",
        ]:
            selected_runtime = summary[f"{key}/total_selected_runtime"]
            otimal_runtime = summary[f"{key}/total_optimal_runtime"]

            selected_runtimes.append(selected_runtime)
            slowdowns.append(selected_runtime / otimal_runtime)
    except KeyError:
        print(f"Key not found: {key} in run {run.id}")
        continue

    runs_adjusted_ctr += 1

    run_id = run.id

    # continue run
    wandb_run = wandb.init(entity="learneddb", project="lcm", id=run_id, resume="must")
    wandb_run.summary.update(
        {
            "qo_total/selected_runtime": sum(selected_runtimes),
            "qo_total/avg_slowdown": sum(slowdowns) / len(slowdowns),
        }
    )

print(f"Adjusted {runs_adjusted_ctr} runs")

In [ ]:
keys = run.summary.keys()
keys = [k for k in keys if "total_selected_runtime" in k]
print(keys)